In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Reproducibility
SEED = 42
np.random.seed(SEED)

In [2]:
# Channel configuration and ground truth
CHANNELS = {
    'paid_search': {
        'true_lift': 0.40,
        'weekly_budget': 15000
    },
    'paid_social': {
        'true_lift': 0.25,
        'weekly_budget': 10000
    },
    'display': {
        'true_lift': 0.05,
        'weekly_budget': 5000
    },
    'affiliate': {
        'true_lift': 0.15,
        'weekly_budget': 7000
    },
    'organic': {
        'true_lift': 0.10,
        'weekly_budget': 0  # no paid spend
    }
}

TOTAL_WEEKS = 104  # 2 years of data
BASE_WEEKLY_REVENUE = 50000  # baseline revenue without any marketing

In [3]:
# ── Weekly Aggregated Dataset (MMM) ──────────────────────────────────────────

weeks = pd.date_range(start='2022-01-03', periods=TOTAL_WEEKS, freq='W-MON')
channel_names = list(CHANNELS.keys())

# Spend per channel with realistic weekly variation
spend_data = {}
for ch, cfg in CHANNELS.items():
    if cfg['weekly_budget'] == 0:
        spend_data[ch + '_spend'] = np.zeros(TOTAL_WEEKS)
    else:
        noise = np.random.normal(1.0, 0.15, TOTAL_WEEKS)
        noise = np.clip(noise, 0.5, 1.5)
        spend_data[ch + '_spend'] = cfg['weekly_budget'] * noise

# Seasonality index (peaks in Dec, dips in Jan-Feb)
week_of_year = np.array([w.isocalendar()[1] for w in weeks])
seasonality = 1 + 0.3 * np.sin(2 * np.pi * (week_of_year - 10) / 52)

# Revenue = baseline + channel contributions + seasonality + noise
revenue = BASE_WEEKLY_REVENUE * seasonality
for ch, cfg in CHANNELS.items():
    ch_spend = spend_data[ch + '_spend']
    revenue += ch_spend * cfg['true_lift']

revenue *= np.random.normal(1.0, 0.05, TOTAL_WEEKS)

# Assemble dataframe
df_mmm = pd.DataFrame({'week': weeks, **spend_data, 'revenue': revenue.round(2)})

print(df_mmm.shape)
print(df_mmm.head())

(104, 7)
        week  paid_search_spend  paid_social_spend  display_spend  \
0 2022-01-03       16117.606844        9758.071433    5386.285765   
1 2022-01-10       14688.905322       10606.076285    7500.000000   
2 2022-01-17       16457.299211       12829.278852    5428.167883   
3 2022-01-24       18426.817177       10261.866719    5851.674230   
4 2022-01-31       14473.154907       10386.325586    5715.501323   

   affiliate_spend  organic_spend   revenue  
0      7866.687166            0.0  50703.44  
1      7854.185118            0.0  48011.54  
2      8370.752748            0.0  53043.13  
3      7022.054034            0.0  51508.55  
4      7716.050620            0.0  56594.70  


In [4]:
df_mmm.to_csv('../data/processed/df_mmm.csv', index=False)
print("Saved: df_mmm.csv")

Saved: df_mmm.csv


In [5]:
# ── User Journey Dataset (MTA) ────────────────────────────────────────────────

N_USERS = 10000
start_date = pd.Timestamp('2022-01-03')
end_date = pd.Timestamp('2023-12-25')

# Transition probabilities between channels
# Each row = current state, values = probability of next state
# 'convert' and 'null' are terminal states
TRANSITIONS = {
    'start': {
        'paid_search': 0.30,
        'paid_social': 0.25,
        'display':     0.25,
        'affiliate':   0.12,
        'organic':     0.08
    },
    'paid_search': {
        'convert':     0.15,
        'paid_social': 0.22,
        'organic':     0.18,
        'display':     0.12,
        'affiliate':   0.10,
        'null':        0.23  
    },
    'paid_social': {
        'convert':     0.09,
        'paid_search': 0.27,
        'display':     0.22,
        'affiliate':   0.15,
        'organic':     0.12,
        'null':        0.15
    },
    'display': {
        'convert':     0.01,
        'paid_search': 0.37,
        'paid_social': 0.22,
        'affiliate':   0.12,
        'organic':     0.12,
        'null':        0.16
    },
    'affiliate': {
        'convert':     0.06,
        'paid_search': 0.27,
        'paid_social': 0.22,
        'display':     0.17,
        'organic':     0.12,
        'null':        0.16
    },
    'organic': {
        'convert':     0.05,
        'paid_search': 0.32,
        'paid_social': 0.22,
        'display':     0.17,
        'affiliate':   0.12,
        'null':        0.12
    }
}

def simulate_journey(max_steps=8):
    """Simulate one user journey through channel transitions."""
    path = []
    state = 'start'
    
    for _ in range(max_steps):
        next_states = TRANSITIONS.get(state, {'null': 1.0})
        next_state = np.random.choice(
            list(next_states.keys()),
            p=list(next_states.values())
        )
        
        if next_state in ('convert', 'null'):
            converted = 1 if next_state == 'convert' else 0
            return path, converted
        
        path.append(next_state)
        state = next_state
    
    # Max steps reached without terminal state
    return path, 0

# Generate journeys
records = []
days_range = (end_date - start_date).days

for user_id in range(N_USERS):
    path, converted = simulate_journey()
    
    if not path:
        path = ['organic']
    
    # Assign dates
    first_touch = start_date + pd.Timedelta(
        days=int(np.random.randint(0, days_range - 7))
    )
    
    for i, channel in enumerate(path):
        date = first_touch + pd.Timedelta(days=i)
        is_last = (i == len(path) - 1)
        records.append({
            'user_id':        user_id,
            'touchpoint_date': date,
            'channel':        channel,
            'conversion':     1 if (is_last and converted) else 0
        })

df_mta = pd.DataFrame(records)
df_mta = df_mta.sort_values(
    ['user_id', 'touchpoint_date']
).reset_index(drop=True)

print(df_mta.shape)
print(df_mta.head(10))
print(f"\nConversion rate: {df_mta[df_mta['conversion']==1].shape[0] / N_USERS:.1%}")
print(f"Total touchpoints: {len(df_mta)}")

(35692, 4)
   user_id touchpoint_date      channel  conversion
0        0      2023-07-15      display           0
1        0      2023-07-16  paid_search           0
2        0      2023-07-17    affiliate           0
3        0      2023-07-18  paid_search           0
4        0      2023-07-19  paid_social           0
5        0      2023-07-20      display           0
6        0      2023-07-21  paid_social           0
7        0      2023-07-22  paid_search           0
8        1      2022-09-18      display           0
9        1      2022-09-19      organic           0

Conversion rate: 27.8%
Total touchpoints: 35692


In [6]:
df_mta.to_csv('../data/processed/df_mta.csv', index=False)
print("Saved: df_mta.csv")

Saved: df_mta.csv


In [7]:
# ── Geo Dataset (Incrementality) ─────────────────────────────────────────────

N_REGIONS = 20
EXPERIMENT_START_WEEK = 78  # experiment starts at week 78 of 104
TEST_CHANNEL = 'paid_social'
TEST_CHANNEL_LIFT = CHANNELS[TEST_CHANNEL]['true_lift']  # 0.25 — our ground truth

regions = [f'region_{i:02d}' for i in range(N_REGIONS)]
control_regions = regions[:10]   # paid_social turned OFF
test_regions = regions[10:]      # business as usual

geo_records = []

for region in regions:
    is_control = region in control_regions
    
    # Each region has its own baseline (size variation)
    region_baseline = BASE_WEEKLY_REVENUE * np.random.uniform(0.7, 1.3)
    
    for week_idx, week in enumerate(weeks):
        in_experiment = week_idx >= EXPERIMENT_START_WEEK
        paid_social_active = not (is_control and in_experiment)
        
        # Seasonality (same as MMM)
        week_num = week.isocalendar()[1]
        season = 1 + 0.3 * np.sin(2 * np.pi * (week_num - 10) / 52)
        
        # Revenue
        rev = region_baseline * season
        
        # Add paid_social contribution only if active
        paid_social_spend = CHANNELS[TEST_CHANNEL]['weekly_budget']
        if paid_social_active:
            rev += paid_social_spend * TEST_CHANNEL_LIFT
        
        # Random noise
        rev *= np.random.normal(1.0, 0.08)
        
        geo_records.append({
            'region': region,
            'week': week,
            'group': 'control' if is_control else 'test',
            'paid_social_active': int(paid_social_active),
            'revenue': round(rev, 2)
        })

df_geo = pd.DataFrame(geo_records)

print(df_geo.shape)
print(df_geo.head(10))
print(f"\nControl regions: {control_regions[:3]}...")
print(f"Experiment starts: week {EXPERIMENT_START_WEEK} ({weeks[EXPERIMENT_START_WEEK].date()})")

(2080, 5)
      region       week    group  paid_social_active   revenue
0  region_00 2022-01-03  control                   1  42788.73
1  region_00 2022-01-10  control                   1  44100.57
2  region_00 2022-01-17  control                   1  42246.72
3  region_00 2022-01-24  control                   1  43961.88
4  region_00 2022-01-31  control                   1  41834.60
5  region_00 2022-02-07  control                   1  47050.45
6  region_00 2022-02-14  control                   1  43813.13
7  region_00 2022-02-21  control                   1  47444.99
8  region_00 2022-02-28  control                   1  44165.67
9  region_00 2022-03-07  control                   1  49651.23

Control regions: ['region_00', 'region_01', 'region_02']...
Experiment starts: week 78 (2023-07-03)


In [8]:
df_geo.to_csv('../data/processed/df_geo.csv', index=False)
print("Saved: df_geo.csv")

Saved: df_geo.csv


In [9]:
# Verify all three datasets
df_mmm_check = pd.read_csv('../data/processed/df_mmm.csv')
df_mta_check = pd.read_csv('../data/processed/df_mta.csv')
df_geo_check = pd.read_csv('../data/processed/df_geo.csv')

print("=== df_mmm ===")
print(f"Shape: {df_mmm_check.shape}")
print(f"Weeks: {df_mmm_check['week'].min()} → {df_mmm_check['week'].max()}")

print("\n=== df_mta ===")
print(f"Shape: {df_mta_check.shape}")
print(f"Users: {df_mta_check['user_id'].nunique()}")
print(f"Conversion rate: {df_mta_check[df_mta_check['conversion']==1]['user_id'].nunique() / df_mta_check['user_id'].nunique():.1%}")

print("\n=== df_geo ===")
print(f"Shape: {df_geo_check.shape}")
print(f"Regions: {df_geo_check['region'].nunique()}")
print(f"Groups: {df_geo_check['group'].unique()}")

print("\n✅ All datasets loaded successfully")

=== df_mmm ===
Shape: (104, 7)
Weeks: 2022-01-03 → 2023-12-25

=== df_mta ===
Shape: (35692, 4)
Users: 10000
Conversion rate: 27.8%

=== df_geo ===
Shape: (2080, 5)
Regions: 20
Groups: <ArrowStringArray>
['control', 'test']
Length: 2, dtype: str

✅ All datasets loaded successfully
